# Stage A — Analysis & Evaluation

Hanging block: **both Newton solvers** (XPBD positional + explicit/SemiImplicit)
vs FEniCSx (FEM tet/hex) vs the analytic 1-D bar. The analytic solution is used
not only to rank the solvers but to **check the FEM's own validity** (§1b).
Evaluates deflection, displacement profile, internal (strain) energy, released
gravitational PE, settling energetics, computation time, Jacobian, dissipation,
equilibrium residual and the differentiable stiffness fit.

Produce the explicit-solver data with `python -m newton_run.run_stage_a --solver semi_implicit`.
Convergence (h-refinement / iteration sweeps) lives in `30_convergence_analysis.ipynb`.

In [ ]:
%matplotlib inline
import os
import numpy as np
import matplotlib.pyplot as plt
from common import params
from compare import energies as en

newton = np.load(params.NEWTON_NPZ)                  # Newton XPBD (positional)
semi   = np.load(params.NEWTON_SEMI_NPZ) if os.path.exists(params.NEWTON_SEMI_NPZ) else None  # Newton explicit
fem    = np.load(params.FEM_NPZ)                      # FEM tet (node-for-node)
hexfem = np.load(params.FEM_HEX_NPZ) if os.path.exists(params.FEM_HEX_NPZ) else None
print(params.summary())
print('Newton explicit (SemiImplicit) run present:', semi is not None)

## 1. Deflection (tip vertical displacement)

In [ ]:
def tip_drop(d):
    free = np.setdiff1d(np.arange(len(d['rest_q'])), d['fixed_nodes'])
    return -(d['final_q'] - d['rest_q'])[free, 2].min() * 1000.0

z_top = newton['rest_q'][newton['fixed_nodes'], 2].mean()
vals = {'Newton XPBD': tip_drop(newton)}
if semi is not None:
    vals['Newton explicit'] = tip_drop(semi)
vals['FEM tet'] = tip_drop(fem)
if hexfem is not None:
    vals['FEM hex'] = tip_drop(hexfem)
vals['analytic 1-D'] = params.analytic_hanging_displacement(
    newton['rest_q'][:, 2].min(), z_top, params.BLOCK_LZ) * 1000.0
for k, v in vals.items():
    print(f'{k:16s}: {v:7.2f} mm')
plt.figure(figsize=(7, 4))
colors = ['tab:orange', 'tab:red', 'tab:blue', 'tab:green', 'grey']
plt.bar(list(vals), list(vals.values()), color=colors[:len(vals)])
plt.ylabel('tip drop [mm]'); plt.title('Stage A - deflection'); plt.xticks(rotation=15)
plt.tight_layout(); plt.show()

## 1b. Is the FEM itself any good? — validity check vs the analytic bar

The analytic 1-D self-weight bar (`u_tip = rho g L^2 / 2E`) is a **sanity anchor**,
not ground truth: it ignores Poisson contraction (here nu=0.42) and 3-D effects,
so we expect the FEM to be *close*, not identical. Linear tets **lock** (come out
too stiff); Hex8 is less prone to it and is the more trustworthy reference. A FEM
that lands within a sensible margin of the analytic value — and converges towards
it as the lateral confinement is relaxed — is "tauglich" (adequate). The h-limit
is quantified in `30_convergence_analysis.ipynb`.

In [ ]:
def prof(d, **kw):
    plt.scatter(d['rest_q'][:, 2], -(d['final_q'] - d['rest_q'])[:, 2] * 1000.0, s=8, alpha=0.4, **kw)
plt.figure(figsize=(6, 5))
prof(newton, label='Newton XPBD', color='tab:orange')
if semi is not None:
    prof(semi, label='Newton explicit', color='tab:red')
prof(fem, label='FEM tet', color='tab:blue')
if hexfem is not None:
    prof(hexfem, label='FEM hex', color='tab:green')
za = np.linspace(newton['rest_q'][:, 2].min(), z_top, 100)
plt.plot(za, params.analytic_hanging_displacement(za, z_top, params.BLOCK_LZ) * 1000.0, 'k--', label='analytic')
plt.xlabel('height z [m]'); plt.ylabel('downward u_z [mm]'); plt.legend(); plt.grid(alpha=0.3)
plt.title('Stage A - displacement profile'); plt.show()

## 2. Displacement profile (downward u_z vs height)

In [ ]:
tets = newton['tet_indices']
m = en.node_masses(newton['rest_q'], tets)
U = {'Newton XPBD': en.strain_energy(newton['rest_q'], newton['final_q'], tets)}
if semi is not None:
    U['Newton explicit'] = en.strain_energy(semi['rest_q'], semi['final_q'], tets)
U['FEM tet'] = en.strain_energy(fem['rest_q'], fem['final_q'], tets)
U['analytic'] = en.analytic_hanging_strain_energy()
dPE = {'Newton': en.gravitational_pe(newton['rest_q'], m) - en.gravitational_pe(newton['final_q'], m),
       'FEM tet': en.gravitational_pe(fem['rest_q'], m) - en.gravitational_pe(fem['final_q'], m)}
for k in U: print(f'strain energy  {k:16s}: {U[k]:.4g} J')
for k in dPE: print(f'released PE    {k:16s}: {dPE[k]:.4g} J')
plt.figure(figsize=(7, 4))
plt.bar(list(U), list(U.values()), color=['tab:orange', 'tab:red', 'tab:blue', 'grey'][:len(U)])
plt.ylabel('strain energy [J]'); plt.title('Stage A - internal energy'); plt.tight_layout(); plt.show()

## 3. Internal (strain) energy and energy balance

Strain energy uses the same Neo-Hookean density on the same tet mesh for Newton
and FEM, so the numbers are directly comparable. We also report the released
gravitational potential energy.

In [ ]:
tets = newton['tet_indices']
m = en.node_masses(newton['rest_q'], tets)
U = {'Newton': en.strain_energy(newton['rest_q'], newton['final_q'], tets),
     'FEM tet': en.strain_energy(fem['rest_q'], fem['final_q'], tets),
     'analytic': en.analytic_hanging_strain_energy()}
dPE = {'Newton': en.gravitational_pe(newton['rest_q'], m) - en.gravitational_pe(newton['final_q'], m),
       'FEM tet': en.gravitational_pe(fem['rest_q'], m) - en.gravitational_pe(fem['final_q'], m)}
for k in U: print(f'strain energy  {k:8s}: {U[k]:.4g} J')
for k in dPE: print(f'released PE    {k:8s}: {dPE[k]:.4g} J')
plt.figure(figsize=(6, 4))
plt.bar(list(U), list(U.values()), color=['tab:orange','tab:blue','grey'])
plt.ylabel('strain energy [J]'); plt.title('Stage A - internal energy'); plt.tight_layout(); plt.show()

## 4. Newton settling energetics (kinetic energy decays as it settles)

In [ ]:
times = {'Newton XPBD': float(newton['wall_time'])}
if semi is not None and 'wall_time' in semi.files:
    times['Newton explicit'] = float(semi['wall_time'])
times['FEM tet'] = float(fem['wall_time'])
if hexfem is not None and 'wall_time' in hexfem.files:
    times['FEM hex'] = float(hexfem['wall_time'])
for k, v in times.items():
    print(f'{k:16s}: {v:8.3f} s')
plt.figure(figsize=(7, 4))
plt.bar(list(times), list(times.values()),
        color=['tab:orange', 'tab:red', 'tab:blue', 'tab:green'][:len(times)])
plt.ylabel('solve wall time [s]'); plt.title('Stage A - computation time'); plt.xticks(rotation=15)
plt.tight_layout(); plt.show()

## 5. Computation time (wall-clock)

The headline trade-off of the whole project: Newton (fast, GPU) vs. implicit FEM
(accurate). Times are for the solve only (not mesh build / IO).

In [ ]:
times = {'Newton XPBD': float(newton['wall_time']), 'FEM tet': float(fem['wall_time'])}
if hexfem is not None and 'wall_time' in hexfem.files:
    times['FEM hex'] = float(hexfem['wall_time'])
for k, v in times.items():
    print(f'{k:14s}: {v:8.3f} s')
plt.figure(figsize=(6, 4))
plt.bar(list(times), list(times.values()), color=['tab:orange','tab:blue','tab:green'][:len(times)])
plt.ylabel('solve wall time [s]'); plt.title('Stage A - computation time'); plt.xticks(rotation=15)
plt.tight_layout(); plt.show()

## 6. Volume / Jacobian (det F) distribution

Volume preservation & incompressibility handling. J = 1 means volume preserved;
the spread of J shows how each solver distributes volumetric strain.

In [ ]:
Jn = en.jacobians(newton['rest_q'], newton['final_q'], tets)
Jf = en.jacobians(fem['rest_q'], fem['final_q'], tets)
print(f"Newton : J mean={Jn.mean():.4f} min={Jn.min():.4f} max={Jn.max():.4f}  dV={en.volume_change(newton['rest_q'],newton['final_q'],tets)*100:+.2f}%")
print(f"FEM tet: J mean={Jf.mean():.4f} min={Jf.min():.4f} max={Jf.max():.4f}  dV={en.volume_change(fem['rest_q'],fem['final_q'],tets)*100:+.2f}%")
plt.figure(figsize=(6, 4))
plt.hist(Jn, bins=40, alpha=0.6, label='Newton', color='tab:orange')
plt.hist(Jf, bins=40, alpha=0.6, label='FEM tet', color='tab:blue')
plt.axvline(1.0, color='k', ls='--', lw=1)
plt.xlabel('per-tet J = det F'); plt.ylabel('count'); plt.legend()
plt.title('Stage A - Jacobian distribution'); plt.show()

## 7. Dissipation & energy budget

Released gravitational PE is stored as strain energy or lost to numerical
dissipation. Newton (dynamic, position-based) damps energy on the way to rest;
the implicit FEM *static* solve jumps straight to equilibrium with no transient,
so it has no dissipation channel. For a suddenly-applied constant load damped to
rest, roughly half the released PE is dissipated.

In [ ]:
rn = en.equilibrium_residual(newton['rest_q'], newton['final_q'], tets, newton['fixed_nodes'])
rf = en.equilibrium_residual(fem['rest_q'], fem['final_q'], tets, fem['fixed_nodes'])
print(f"Newton XPBD : free-node residual RMS = {rn['free_rms']:.4g} N | reaction_z={rn['reaction_z']:.4g} N (weight={rn['weight']:.4g} N)")
rs = None
if semi is not None:
    rs = en.equilibrium_residual(semi['rest_q'], semi['final_q'], tets, semi['fixed_nodes'])
    print(f"Newton expl.: free-node residual RMS = {rs['free_rms']:.4g} N | reaction_z={rs['reaction_z']:.4g} N")
print(f"FEM tet     : free-node residual RMS = {rf['free_rms']:.4g} N | reaction_z={rf['reaction_z']:.4g} N (weight={rf['weight']:.4g} N)")
plt.figure(figsize=(6, 4))
fn = np.linalg.norm(rn['residual'][rn['free']], axis=1)
ff = np.linalg.norm(rf['residual'][rf['free']], axis=1)
plt.hist(fn, bins=40, alpha=0.6, label='Newton XPBD', color='tab:orange')
if rs is not None:
    fs = np.linalg.norm(rs['residual'][rs['free']], axis=1)
    plt.hist(fs, bins=40, alpha=0.6, label='Newton explicit', color='tab:red')
plt.hist(ff, bins=40, alpha=0.6, label='FEM tet', color='tab:blue')
plt.xlabel('per-node out-of-balance force [N]'); plt.ylabel('count'); plt.legend()
plt.title('Stage A - equilibrium residual'); plt.show()

## 8. Equilibrium residual (out-of-balance force)

The most direct 'solve vs project' test: internal elastic force + gravity at each
free node. FEM solved R(u)=0 so its residual is ~0; XPBD's nonzero residual is
how far its settled state is from the true material's static equilibrium. The
clamped-node reaction should equal the total weight.

In [ ]:
rn = en.equilibrium_residual(newton['rest_q'], newton['final_q'], tets, newton['fixed_nodes'])
rf = en.equilibrium_residual(fem['rest_q'], fem['final_q'], tets, fem['fixed_nodes'])
print(f"Newton : free-node residual RMS = {rn['free_rms']:.4g} N | reaction_z={rn['reaction_z']:.4g} N (weight={rn['weight']:.4g} N)")
print(f"FEM tet: free-node residual RMS = {rf['free_rms']:.4g} N | reaction_z={rf['reaction_z']:.4g} N (weight={rf['weight']:.4g} N)")
plt.figure(figsize=(6, 4))
fn = np.linalg.norm(rn['residual'][rn['free']], axis=1)
ff = np.linalg.norm(rf['residual'][rf['free']], axis=1)
plt.hist(fn, bins=40, alpha=0.6, label='Newton', color='tab:orange')
plt.hist(ff, bins=40, alpha=0.6, label='FEM tet', color='tab:blue')
plt.xlabel('per-node out-of-balance force [N]'); plt.ylabel('count'); plt.legend()
plt.title('Stage A - equilibrium residual'); plt.show()

## 9. Differentiable evaluation (Newton's superpower)

Newton is built on differentiable Warp, so we can backpropagate a loss through the
whole simulation. Here the loss is the node-for-node mismatch to the FEM reference,
and we differentiate w.r.t. a stiffness multiplier theta. Two outputs:

* **sensitivity** dLoss/dtheta at theta=1 (exact, one backward pass),
* **fitted theta\***: the effective-stiffness multiplier that makes Newton match
  FEM (theta\*>1 -> Newton effectively softer, theta\*<1 -> stiffer).

Run `python -m newton_run.diffsim_stage_a` (CUDA) to produce the data.

In [ ]:
if os.path.exists(params.DIFFSIM_NPZ):
    ds = np.load(params.DIFFSIM_NPZ)
    print(f"Sensitivity at theta=1 : loss = {float(ds['loss0']):.4e} m^2, dLoss/dtheta = {float(ds['grad0']):.4e}")
    print(f"Fitted multiplier theta* = {float(ds['theta_star']):.4f}  "
          f"({'Newton softer than FEM' if float(ds['theta_star'])>1 else 'Newton stiffer than FEM'})")
    fig, ax1 = plt.subplots(figsize=(6, 4))
    ax1.semilogy(ds['loss_history'], color='tab:red'); ax1.set_xlabel('iteration')
    ax1.set_ylabel('mismatch loss [m^2]', color='tab:red')
    ax2 = ax1.twinx(); ax2.plot(ds['theta_history'], color='tab:blue')
    ax2.set_ylabel('theta (stiffness mult.)', color='tab:blue'); ax2.axhline(1.0, color='grey', ls=':', lw=1)
    plt.title('Stage A - differentiable stiffness fit'); plt.tight_layout(); plt.show()
else:
    print('No diffsim result yet. Run: python -m newton_run.diffsim_stage_a')
    print('(Newton differentiates through the sim to fit the effective stiffness vs FEM.)')